## Full Near-real-time (NRT) pipeline

### Data Loading
* load local Data
* download new timestamp (recent date)
* merge dataset

### Run NRT Detection

In [ ]:
# imports
from pathlib import Path

import xarray as xr
import geopandas as gpd

from water_timeseries.breakpoint import NRTBreakpoint
from water_timeseries.dataset import DWDataset
from water_timeseries.downloader import EarthEngineDownloader

## Load data

In [ ]:
def dw_test_dataset():
    """Load the real Dynamic World test dataset.

    Returns the actual DW dataset from tests/data/lakes_dw_test.nc
    """
    test_data_dir = Path("../tests/data")
    dw_file = test_data_dir / "lakes_dw_test.nc"
    return xr.open_dataset(dw_file)

Load pre loaded historic Dynamic World Dataset

In [ ]:
dw_ds_historical = DWDataset(dw_test_dataset())

Show valid ids and dates
* the last available date is June 2025 (2025-06-01)
* There are 5 lakes in the dataset

In [ ]:
print(dw_ds_historical.object_ids_)
print(dw_ds_historical.dates_[-1])


## Data Download
Now we need to append our timeseries

#### Load and match vector polygons for download
* load vector dataset
* confirm overlapping id_geohash values between ds and vector file

In [ ]:
vector_path = "../tests/data/lakes_polygons_test.parquet"
gdf = gpd.read_parquet(vector_path)

In [ ]:
gdf_filtered = gdf[gdf["id_geohash"].isin(dw_ds_historical.object_ids_)]
gdf_filtered["id_geohash"].tolist()

#### Download data
* first setup the downloader using a dedicated GEE project name
* **Please insert your GEE project name**
* download dynamic world data for the five lakes for July 2025
* the downloaded dataset is a xr.dataset

In [ ]:
ee_project = "YOUR_EE_PROJECT_NAME"
downloader = EarthEngineDownloader(ee_project=ee_project)

In [ ]:
ds_downloaded = downloader.download_dw_monthly(
    vector_dataset=vector_path,
    name_attribute="id_geohash",
    date_list=["2025-07"],
    id_list=gdf_filtered["id_geohash"].tolist(),
)
ds_downloaded

In [ ]:
dw_ds_new = DWDataset(ds_downloaded)

#### Merge datasets

* We will convert it to DWDataset
* both DW dataset (historical + new) can be simply merged

In [ ]:
dw_ds_merged = dw_ds_historical.merge(dw_ds_new)

The datasets were concatenated where the new date was appended,
* checking the latest 2 dates shows that the new date was appended
* object ids (id_geohash / individual lakes) stay the same

In [ ]:
print(dw_ds_merged.dates_[-2:])
print(dw_ds_merged.object_ids_)

Lets visualize if appending the dataset worked

In [ ]:
dw_ds_merged.plot_timeseries_interactive(id_geohash="b7uefy0bvcrc")

### Near Real-time analysis
* NRT analysis uses ARIMA to determine if a specified date (typically the latest date) is an outlier compared to the previous time-series

In [ ]:
bp_nrt = NRTBreakpoint()

Here we can check the ARIMA prediction for all 5 lakes
* water_residual values, which indicate the difference from the predicted water area are all close to zero
* water_predicted is in all cases within the confidence range ( > water_predicted_lower_90, < water_predicted_upper_90 )
* there is no meaningful lake drainage event

In [ ]:
bp_nrtbreaks_simple_dw_2018 = bp_nrt.calculate_break(
    dataset=dw_ds_merged, analysis_date="2025-07", data_aggregation_period="all"
)
bp_nrtbreaks_simple_dw_2018

#### Geospatial postprocessing
* We can join the output on the input vector layer

In [ ]:
# full join keeping all lakes in the gdf, even those without breakpoints (will have NaN values for breakpoint columns)
joined_all_2018 = gdf_filtered.set_index("id_geohash").join(bp_nrtbreaks_simple_dw_2018)
joined_all_2018

#### Filter
* filter to data with significant lake area loss in the defined month
* results in an empty dataframe in our case

In [ ]:
# filter to lakes with a significant negative breakpoint (water loss) in the last month
joined_all_2018.query("water_residual < -0.25")

### Analysis for a know break data
* we know that one lake drained in June 2018 - so let's try this date

In [ ]:
bp_nrtbreaks_simple_dw_2018 = bp_nrt.calculate_break(
    dataset=dw_ds_merged, analysis_date="2018-06", data_aggregation_period="all"
)
# full join keeping all lakes in the gdf, even those without breakpoints (will have NaN values for breakpoint columns)
joined_all_2018 = gdf_filtered.set_index("id_geohash").join(bp_nrtbreaks_simple_dw_2018)

# filter to lakes with a significant negative breakpoint (water loss) in the last month
joined_all_2018.query("water_residual < -0.25")

We can explore the result and save it to a parquet file or other vector format

In [ ]:
# filter relevant columns for output
columns = ["id_geohash"] + bp_nrtbreaks_simple_dw_2018.columns.tolist() + ["geometry"]
# make query and reset index to get id_geohash back as a column, then select relevant columns for output
output_df = joined_all_2018.query("water_residual < -0.25").reset_index()[columns]

In [ ]:
# visualize output
output_df.explore()

In [ ]:
output_df.to_parquet("nrt_breakpoints_lakes_2018-06.parquet")